# 2. Analysing processed data with Raphael

A processed session: spikes from Raphael's processed tree, behaviour from this project.
Raphael loads the neurons and scores their tuning; the project says what the arena is.

This notebook uses a **demo session** (four place cells, two untuned units, one that failed curation).
For your own data, set `ROOT_DATA` and `session` to a session in your processed tree.

In [ ]:
import os
from pathlib import Path

# Run from the repository root, wherever the notebook was opened from.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').exists())
os.chdir(ROOT)

import matplotlib.pyplot as plt
from raphael import plots, style

import starter_project  # noqa: F401  (importing the project applies the labs' figure style)

In [ ]:
from raphael.ephys.layout import Session

from starter_project.demo import make_processed_session

ROOT_DATA = Path('data/processed')
session = Session(mouse='1', day='1', session_type='OF1')
make_processed_session(ROOT_DATA, session)
sorted(p.name for p in session.folder(ROOT_DATA).iterdir())

## Load the neurons

`population.load` takes the files' shared prefix and returns the same object for spikes
(`_clusters.npz`) or calcium (`_traces.npz` + `_rois.csv`). `.accepted()` keeps units that passed curation.

In [ ]:
from raphael import population

pop = population.load(session.folder(ROOT_DATA) / f'{session.stem}_srt-kilosort4')
print(len(pop.ids), 'units,', int(pop.table['accepted'].sum()), 'accepted')
pop = pop.accepted()
pop.table

In [ ]:
fig, ax = plt.subplots(figsize=style.figsize('double', height_mm=35))
plots.raster(ax, pop, start=0, stop=60)
ax.set_title('First minute');

## Behaviour (project code)

How behaviour is stored is the project's choice; `starter_project.behaviour` loads it into the
pynapple objects Raphael's scores read: `P_x`, `P_y`, `S`, `H` and a `moving` epoch.

In [ ]:
from starter_project.behaviour import load_behaviour

beh = load_behaviour(ROOT_DATA, session)
fig, ax = plt.subplots(figsize=style.figsize(50, height_mm=50))
ax.plot(beh['P_x'].values, beh['P_y'].values, lw=0.2, color=style.GREYS['mid'])
ax.set(aspect='equal', title='Path', xlabel='x (cm)', ylabel='y (cm)');

## Rate maps and tuning scores

`score_session` (in `starter_project/analysis.py`) calls Raphael's tuning scores with this
project's `bin_config` (`starter_project/arena.py`); each score is tested against a shuffled null.
`raphael.plots.rate_map` draws each map against the arena's real size, with unvisited bins left blank.

In [ ]:
from starter_project.analysis import score_session
from starter_project.arena import ARENA_CM

scores = score_session(ROOT_DATA, session, n_shuffles=50)

fig, axes = plt.subplots(1, len(scores), figsize=style.figsize('double', height_mm=35))
for ax, row in zip(axes, scores.itertuples()):
    plots.rate_map(ax, pop.data[row.unit], beh, bounds=(0, ARENA_CM[0], 0, ARENA_CM[1]))
    ax.set_title(f'Unit {row.unit}\nSI {row.spatial_information:.2f}' + (' *' if row.si_sig else ''))
scores

## Using Raphael's functions directly

The same scores, one unit at a time, from `raphael.tuning_scores`:

In [ ]:
import numpy as np
from raphael.tuning_scores import compute_spatial_information

from starter_project.arena import bin_config

unit = pop.ids[0]
result, = compute_spatial_information(beh, bin_config(), pop.data[unit], unit,
                                      alpha=0.05, n_shuffles=50)
fig, ax = plt.subplots(figsize=style.figsize(60, height_mm=40))
plots.score_histogram(ax, [result['spatial_information']], null=result['null_distribution'],
                      threshold=np.percentile(result['null_distribution'], 95),
                      label=f'Unit {unit}', xlabel='Spatial information (bits/spike)');

## Next

- Put analysis you want to keep in `src/starter_project/` and call it from a script in `scripts/`
  (see `scripts/analyse_tuning.py`, which saves its figure with `style.save`, as PDF and PNG).
- More standard plots (head direction, theta phase, peri-event, traces): Raphael's docs/plots.md.
- Something other projects would use too? Suggest it for Raphael.